In [11]:
# Import libarary
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import random

In [21]:
# Load data
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = 'pitch_shift.wav'
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

In [13]:
# Export Helper
def export_pydub(audio):
    while True:
        file_name = input("Input filename (.wav): ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be end with .wav")
    audio.export(file_name, format="wav")
    print("audio filename saved as", file_name)
    
def export_librosa(audio):
    while True:
        file_name = input("Input filename (.wav): ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be end with .wav")
    sf.write(file_name, np.clip(audio, -1,1), sr)
    print("audio filename saved as", file_name)

In [14]:
# Predict Text
def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)
    waveform = waveform.mean(dim=0, keepdim=True)
    
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
        
    with torch.inference_mode():
        emission, _ = model(waveform)
        
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0
            
        def forward(self, emission: torch.Tensor) -> str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])
        
    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript
    

In [18]:
# aug : adjvolume, adjpitch, fade_in, fade_out

def adjvolume(clip):
    while True:
        db = float(input("Input DB for adjust volume (Normal is 0) (-30 to 30): "))
        if -30 <= db <= 30:
            break
        else:
            print("must be between -30 to 30")
    export_pydub(clip + db)
    
def adjpitch():
    while True:
        p_rate = float(input("Input pitch rate (-5 to 5): "))
        if -5 <= p_rate <= 5:
            break
        else:
            print("must be between -5 to 5")
    pitch_shift = librosa.effects.pitch_shift(audio, sr=sr, n_steps=p_rate)
    export_librosa(pitch_shift)
    
def fade_in(clip):
    while True:
        duration = float(input("Input fadein duration in ms (1s = 1000ms): "))
        if duration > 0:
            print("must be more than 0")
        else:
            break
    export_pydub(clip.fade_in(duration))
    
def fade_out(clip):
    while True:
        duration = float(input("Input fadeout duration in ms (1s = 1000ms): "))
        if duration > 0:
            print("must be more than 0")
        else:
            break
    export_pydub(clip.fade_out(duration))

In [19]:
def aug_menu():
    while True:
        print("1. Adjust Volume")
        print("2. Adjust Pitch Shift")
        print("3. Fade In")       
        print("4. Fade Out")       
        print("5. Exit")
        
        choice_aug = input(">> ")
        if choice_aug == "1":
            adjvolume(clip)
        elif choice_aug == "2":
            adjpitch()
        elif choice_aug == "3":
            fade_in(clip)
        elif choice_aug == "4":
            fade_out(clip)
        elif choice_aug == "5":
            break
        else:
            print("wrong input must be (1-5)")

In [22]:
def mainmenu():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        
        choice = input(">> ")
        if choice == "1":
            aug_menu()
        elif choice == "2":
            transcript = predict_text(audio_file)
            print("Predict Transcript:")
            print(transcript)
        elif choice == "3":
            print("bye")
            break
        else:
            print("wrong input must be (1-3)")
            
mainmenu()

1. Audio Augmentation
2. Predict Text
3. Exit
Predict Transcript:
AND ON THE TIME SHE AMED TO NA MINE THAT THATTEEN YEARS HAV PASSED SINCE IT WAS FISHED MANY RA SINCE IT WAS BEGUN AN NOT DURING TAT PAN PLACES MATTERS BOOKS ANDAPINIONS HAVE HADET ON CONSIDERABLE CHANGES 
1. Audio Augmentation
2. Predict Text
3. Exit
bye
